In [1]:
import os, glob, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# List all input datasets
print("\n--- Kaggle Input Datasets ---")
for d in sorted(os.listdir('/kaggle/input')):
    print(f"  /kaggle/input/{d}/")

# Verify VOC structure
VOC_ROOT = '/kaggle/input/pascal-voc-2012-dataset/VOCdevkit/VOC2012'
print(f"\n--- VOC2012 at: {VOC_ROOT} ---")
if os.path.exists(VOC_ROOT):
    for item in sorted(os.listdir(VOC_ROOT)):
        path = os.path.join(VOC_ROOT, item)
        count = len(os.listdir(path)) if os.path.isdir(path) else 0
        print(f"  {item}/  ({count} files)" if os.path.isdir(path) else f"  {item}")
else:
    # Auto-detect if path is different
    matches = glob.glob('/kaggle/input/**/JPEGImages', recursive=True)
    if matches:
        detected = os.path.dirname(matches[0])
        print(f"  NOT FOUND at expected path. Detected at: {detected}")
        print(f"  *** Change VOC_ROOT to: {detected} ***")
    else:
        print("  ERROR: Could not find VOC dataset at all!")

PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB

--- Kaggle Input Datasets ---
  /kaggle/input/datasets/

--- VOC2012 at: /kaggle/input/pascal-voc-2012-dataset/VOCdevkit/VOC2012 ---
  NOT FOUND at expected path. Detected at: /kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val
  *** Change VOC_ROOT to: /kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val ***


In [2]:
import os

os.makedirs('/kaggle/working/patnet_project', exist_ok=True)
os.chdir('/kaggle/working/patnet_project')

# Clone PATNet repository
!git clone https://github.com/slei109/PATNet.git
os.chdir('/kaggle/working/patnet_project/PATNet')
print("Current dir:", os.getcwd())
print("Files:", os.listdir('.'))

# Install dependencies (Kaggle already has PyTorch, just need tensorboardX)
!pip install tensorboardX -q
print("\nDependencies installed.")

Cloning into 'PATNet'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 145 (delta 64), reused 59 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 2.78 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Current dir: /kaggle/working/patnet_project/PATNet
Files: ['train.py', 'data', 'test.py', 'README.md', 'common', '.git', 'model']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.9 MB/s eta 0:00:00

Dependencies installed.


In [3]:
import os

os.chdir('/kaggle/working/patnet_project/PATNet')

# This is the critical file PATNet needs for training
# Google Drive file ID from PATNet README
FILE_ID = "10zxG2VExoEZUeyQl_uXga2OWHjGeZaf2"

!pip install gdown -q

# Download the SBD augmented segmentation masks
!gdown "https://drive.google.com/uc?id=10zxG2VExoEZUeyQl_uXga2OWHjGeZaf2" -O /kaggle/working/SegmentationClassAug.zip

# Verify download
import os
size = os.path.getsize('/kaggle/working/SegmentationClassAug.zip')
print(f"Downloaded: {size/1e6:.1f} MB")

Downloading...
From (original): https://drive.google.com/uc?id=10zxG2VExoEZUeyQl_uXga2OWHjGeZaf2
From (redirected): https://drive.google.com/uc?id=10zxG2VExoEZUeyQl_uXga2OWHjGeZaf2&confirm=t&uuid=1daa030e-acb4-40c0-b627-c2e765f192f4
To: /kaggle/working/SegmentationClassAug.zip
100%|██████████████████████████████████████| 27.1M/27.1M [00:01<00:00, 23.8MB/s]
Downloaded: 27.1 MB


In [4]:
import os, zipfile

# Extract the SBD masks
print("Extracting SBD masks...")
with zipfile.ZipFile('/kaggle/working/SegmentationClassAug.zip', 'r') as z:
    z.extractall('/kaggle/working/sbd_extract/')

# Check what was extracted
for root, dirs, files in os.walk('/kaggle/working/sbd_extract/'):
    level = root.replace('/kaggle/working/sbd_extract/', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level >= 2:
        break

Extracting SBD masks...
/ (0 files)
SegmentationClassAug/ (12031 files)


In [14]:
import os, glob

VOC_INPUT = '/kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val'
VOC_WORK = '/kaggle/working/VOC2012'
os.makedirs(VOC_WORK, exist_ok=True)

# Symlink all VOC folders
for folder in ['JPEGImages', 'SegmentationClass', 'ImageSets', 'Annotations']:
    src = os.path.join(VOC_INPUT, folder)
    dst = os.path.join(VOC_WORK, folder)
    if os.path.exists(dst):
        print(f"Already exists: {folder}")
    elif os.path.exists(src):
        os.symlink(src, dst)
        print(f"Linked: {folder}  ({len(os.listdir(src))} files)")
    else:
        print(f"ERROR: {folder} not found")

# SegmentationClassAug is directly at /kaggle/working/sbd_extract/SegmentationClassAug
seg_aug_src = '/kaggle/working/sbd_extract/SegmentationClassAug'
seg_aug_dst = os.path.join(VOC_WORK, 'SegmentationClassAug')

if os.path.exists(seg_aug_dst):
    print(f"Already exists: SegmentationClassAug ({len(os.listdir(seg_aug_dst))} files)")
elif os.path.exists(seg_aug_src):
    os.symlink(seg_aug_src, seg_aug_dst)
    print(f"Linked: SegmentationClassAug ({len(os.listdir(seg_aug_src))} files)")
else:
    print("ERROR: SegmentationClassAug not found at expected path")

# Final check
print("\n=== Final VOC_WORK structure ===")
for item in sorted(os.listdir(VOC_WORK)):
    p = os.path.join(VOC_WORK, item)
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {item}/  ({n} files)")

Linked: JPEGImages  (17125 files)
Linked: SegmentationClass  (2913 files)
Linked: ImageSets  (4 files)
Linked: Annotations  (17125 files)
Already exists: SegmentationClassAug (12031 files)

=== Final VOC_WORK structure ===
  Annotations/  (17125 files)
  ImageSets/  (4 files)
  JPEGImages/  (17125 files)
  SegmentationClass/  (2913 files)
  SegmentationClassAug/  (12031 files)


In [15]:
import os

os.chdir('/kaggle/working')
!git clone https://github.com/slei109/PATNet.git
os.chdir('/kaggle/working/PATNet')
print("Files:", os.listdir('.'))

Cloning into 'PATNet'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 145 (delta 64), reused 59 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 2.78 MiB | 16.26 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Files: ['train.py', 'data', 'test.py', 'README.md', 'common', '.git', 'model']


In [16]:
import os
os.chdir('/kaggle/working/PATNet')
!grep -rn "VOC2012\|datapath\|data_root\|data_path" --include="*.py" .

./train.py:58:    parser.add_argument('--datapath', type=str, default='../VOCdevkit')
./train.py:85:    FSSDataset.initialize(img_size=400, datapath=args.datapath)
./train.py:87:    FSSDataset.initialize(img_size=400, datapath='path_to_your_dataset')
./data/lung.py:13:    def __init__(self, datapath, fold, transform, split, shot, num=600):
./data/lung.py:19:        self.base_path = os.path.join(datapath, 'LungSegmentation')
./data/isic.py:13:    def __init__(self, datapath, fold, transform, split, shot, num=600):
./data/isic.py:19:        self.base_path = os.path.join(datapath, 'ISIC')
./data/dataset.py:15:    def initialize(cls, img_size, datapath):
./data/dataset.py:27:        cls.datapath = datapath
./data/dataset.py:40:        dataset = cls.datasets[benchmark](cls.datapath, fold=fold, transform=cls.transform, split=split, shot=shot)
./data/fss.py:13:    def __init__(self, datapath, fold, transform, split, shot):
./data/fss.py:18:        self.base_path = os.path.join(datapath, 'FSS-

In [17]:
import os
os.chdir('/kaggle/working/PATNet')

# --- Fix 1: train.py default datapath ---
with open('train.py', 'r') as f:
    content = f.read()

content = content.replace(
    "default='../VOCdevkit'",
    "default='/kaggle/working/VOC2012'"
)
# Also fix the hardcoded path on line 87
content = content.replace(
    "datapath='path_to_your_dataset'",
    "datapath='/kaggle/working/VOC2012'"
)

with open('train.py', 'w') as f:
    f.write(content)
print("✓ train.py patched")

# --- Fix 2: data/pascal.py hardcoded paths ---
with open('data/pascal.py', 'r') as f:
    content = f.read()

content = content.replace(
    "self.img_path = os.path.join('../VOCdevkit/VOC2012/JPEGImages/')",
    "self.img_path = os.path.join('/kaggle/working/VOC2012/JPEGImages/')"
)
content = content.replace(
    "self.ann_path = os.path.join('../VOCdevkit/VOC2012/SegmentationClassAug/')",
    "self.ann_path = os.path.join('/kaggle/working/VOC2012/SegmentationClassAug/')"
)

with open('data/pascal.py', 'w') as f:
    f.write(content)
print("✓ data/pascal.py patched")

# --- Verify both patches ---
print("\n=== train.py datapath lines ===")
!grep -n "datapath\|VOC" train.py

print("\n=== data/pascal.py path lines ===")
!grep -n "img_path\|ann_path\|VOC" data/pascal.py


✓ train.py patched
✓ data/pascal.py patched

=== train.py datapath lines ===
58:    parser.add_argument('--datapath', type=str, default='/kaggle/working/VOC2012')
85:    FSSDataset.initialize(img_size=400, datapath=args.datapath)
87:    FSSDataset.initialize(img_size=400, datapath='/kaggle/working/VOC2012')

=== data/pascal.py path lines ===
20:        self.img_path = os.path.join('/kaggle/working/VOC2012/JPEGImages/')
21:        self.ann_path = os.path.join('/kaggle/working/VOC2012/SegmentationClassAug/')
87:        mask = torch.tensor(np.array(Image.open(os.path.join(self.ann_path, img_name) + '.png')))
92:        return Image.open(os.path.join(self.img_path, img_name) + '.jpg')


In [18]:
import os
os.chdir('/kaggle/working/PATNet')
!pip install tensorboardX -q
print("✓ tensorboardX installed")

✓ tensorboardX installed


In [23]:
import os
os.chdir('/kaggle/working/PATNet')

# Fix deprecated 'pretrained' argument in model files
for fname in ['model/patnet.py', 'common/backbone.py', 'model/backbone.py']:
    if os.path.exists(fname):
        with open(fname, 'r') as f:
            content = f.read()
        # Fix pretrained=True -> weights='IMAGENET1K_V1'
        new = content.replace("pretrained=True", "pretrained=False")
        with open(fname, 'w') as f:
            f.write(new)
        print(f"✓ Fixed {fname}")

!grep -rn "pretrained" model/ common/ --include="*.py"

✓ Fixed model/patnet.py
model/patnet.py:23:            self.backbone = vgg.vgg16(pretrained=False)
model/patnet.py:37:            self.backbone = resnet.resnet50(pretrained=False)


In [26]:
os.chdir('/kaggle/working/PATNet')

new_train = '''r""" PATNet training (validation) code """
import sys
sys.path.insert(0, "../")

import argparse
import torch.optim as optim
import torch.nn as nn
import torch

from model.patnet import PATNetwork
from common.logger import Logger, AverageMeter
from common.evaluation import Evaluator
from common import utils
from data.dataset import FSSDataset


def train(epoch, model, dataloader, optimizer, training):
    r""" Train PATNet """
    utils.fix_randseed(None) if training else utils.fix_randseed(0)
    model.train_mode() if training else model.eval()
    average_meter = AverageMeter(dataloader.dataset)

    for idx, batch in enumerate(dataloader):
        batch = utils.to_cuda(batch)
        logit_mask = model(batch[\'query_img\'], batch[\'support_imgs\'].squeeze(1), batch[\'support_masks\'].squeeze(1))
        pred_mask = logit_mask.argmax(dim=1)

        loss = model.compute_objective(logit_mask, batch[\'query_mask\'])
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        area_inter, area_union = Evaluator.classify_prediction(pred_mask, batch)
        average_meter.update(area_inter, area_union, batch[\'class_id\'], loss.detach().clone())
        average_meter.write_process(idx, len(dataloader), epoch, write_batch_idx=10)

    average_meter.write_result(\'Training\' if training else \'Validation\', epoch)
    avg_loss = utils.mean(average_meter.loss_buf)
    miou, fb_iou = average_meter.compute_iou()
    return avg_loss, miou, fb_iou


if __name__ == \'__main__\':
    parser = argparse.ArgumentParser(description=\'PATNet Training\')
    parser.add_argument(\'--datapath\', type=str, default=\'/kaggle/working/VOC2012\')
    parser.add_argument(\'--benchmark\', type=str, default=\'pascal\')
    parser.add_argument(\'--logpath\', type=str, default=\'patnet_fold0\')
    parser.add_argument(\'--bsz\', type=int, default=12)
    parser.add_argument(\'--lr\', type=float, default=1e-3)
    parser.add_argument(\'--niter\', type=int, default=200)
    parser.add_argument(\'--nworker\', type=int, default=2)
    parser.add_argument(\'--fold\', type=int, default=0, choices=[0, 1, 2, 3, 4])
    parser.add_argument(\'--backbone\', type=str, default=\'resnet50\', choices=[\'vgg16\', \'resnet50\'])
    args = parser.parse_args()
    Logger.initialize(args, training=True)

    # Model initialization — no DataParallel, single GPU
    model = PATNetwork(args.backbone)
    Logger.log_params(model)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    Logger.info(\'# available GPUs: %d\' % torch.cuda.device_count())
    model.to(device)

    optimizer = optim.Adam([{"params": model.parameters(), "lr": args.lr}])
    Evaluator.initialize()

    # Dataset — train on pascal, validate on pascal (same fold)
    FSSDataset.initialize(img_size=400, datapath=args.datapath)
    dataloader_trn = FSSDataset.build_dataloader(args.benchmark, args.bsz, args.nworker, args.fold, \'trn\')
    dataloader_val = FSSDataset.build_dataloader(args.benchmark, args.bsz, args.nworker, args.fold, \'val\')

    best_val_miou = float(\'-inf\')
    for epoch in range(args.niter):
        trn_loss, trn_miou, trn_fb_iou = train(epoch, model, dataloader_trn, optimizer, training=True)
        with torch.no_grad():
            val_loss, val_miou, val_fb_iou = train(epoch, model, dataloader_val, optimizer, training=False)

        if val_miou > best_val_miou:
            best_val_miou = val_miou
            Logger.save_model_miou(model, epoch, val_miou)
            print(f"  >> New best model saved! val_mIoU: {val_miou:.2f}")

        Logger.tbd_writer.add_scalars(\'data/loss\', {\'trn_loss\': trn_loss, \'val_loss\': val_loss}, epoch)
        Logger.tbd_writer.add_scalars(\'data/miou\', {\'trn_miou\': trn_miou, \'val_miou\': val_miou}, epoch)
        Logger.tbd_writer.add_scalars(\'data/fb_iou\', {\'trn_fb_iou\': trn_fb_iou, \'val_fb_iou\': val_fb_iou}, epoch)
        Logger.tbd_writer.flush()

    Logger.tbd_writer.close()
    Logger.info(\'==================== Finished Training ====================\')
'''

with open('train.py', 'w') as f:
    f.write(new_train)
print("✓ train.py rewritten")

# Verify
!grep -n "DataParallel\|fss\|niter\|dataloader_val\|benchmark" train.py

✓ train.py rewritten
47:    parser.add_argument('--benchmark', type=str, default='pascal')
51:    parser.add_argument('--niter', type=int, default=200)
58:    # Model initialization — no DataParallel, single GPU
70:    dataloader_trn = FSSDataset.build_dataloader(args.benchmark, args.bsz, args.nworker, args.fold, 'trn')
71:    dataloader_val = FSSDataset.build_dataloader(args.benchmark, args.bsz, args.nworker, args.fold, 'val')
74:    for epoch in range(args.niter):
77:            val_loss, val_miou, val_fb_iou = train(epoch, model, dataloader_val, optimizer, training=False)


In [34]:
import os, glob

# Find ALL .pt files anywhere in working directory
print("=== All .pt files ===")
for f in glob.glob('/kaggle/working/**/*.pt', recursive=True):
    size = os.path.getsize(f)/1e6
    print(f"{f}  ({size:.1f} MB)")

# Find all folders created
print("\n=== All folders in /kaggle/working ===")
for item in os.listdir('/kaggle/working'):
    print(f"  {item}")

# Find the actual log folder name
print("\n=== All folders inside PATNet ===")
for item in os.listdir('/kaggle/working/PATNet'):
    print(f"  {item}")

=== All .pt files ===
/kaggle/working/PATNet/logs/final_fold0.log/best_model.pt  (112.9 MB)

=== All folders in /kaggle/working ===
  patnet_project
  sbd_extract
  .virtual_documents
  SegmentationClassAug.zip
  PATNet
  VOC2012

=== All folders inside PATNet ===
  train.py
  data
  test.py
  README.md
  common
  .git
  logs
  model


In [36]:
# Check the backbone file for pretrained issue
os.chdir('/kaggle/working/PATNet')
!grep -rn "pretrained" model/ common/ --include="*.py"

import os
os.chdir('/kaggle/working/PATNet')

with open('model/patnet.py', 'r') as f:
    content = f.read()

# Fix pretrained using new torchvision weights API
content = content.replace(
    "self.backbone = vgg.vgg16(pretrained=False)",
    "self.backbone = vgg.vgg16(weights='VGG16_Weights.IMAGENET1K_V1')"
)
content = content.replace(
    "self.backbone = resnet.resnet50(pretrained=False)",
    "self.backbone = resnet.resnet50(weights='ResNet50_Weights.IMAGENET1K_V1')"
)

with open('model/patnet.py', 'w') as f:
    f.write(content)

print("✓ Fixed pretrained weights")
!grep -n "pretrained\|weights" model/patnet.py

model/patnet.py:23:            self.backbone = vgg.vgg16(pretrained=False)
model/patnet.py:37:            self.backbone = resnet.resnet50(pretrained=False)
✓ Fixed pretrained weights
23:            self.backbone = vgg.vgg16(weights='VGG16_Weights.IMAGENET1K_V1')
37:            self.backbone = resnet.resnet50(weights='ResNet50_Weights.IMAGENET1K_V1')


In [40]:
import os
os.chdir('/kaggle/working/PATNet')

with open('model/patnet.py', 'r') as f:
    content = f.read()

# Fix the weights - use the proper import and enum
old_resnet = "self.backbone = resnet.resnet50(weights='ResNet50_Weights.IMAGENET1K_V1')"
new_resnet = "self.backbone = resnet.resnet50(weights=resnet.ResNet50_Weights.IMAGENET1K_V1)"

old_vgg = "self.backbone = vgg.vgg16(weights='VGG16_Weights.IMAGENET1K_V1')"
new_vgg = "self.backbone = vgg.vgg16(weights=vgg.VGG16_Weights.IMAGENET1K_V1)"

content = content.replace(old_resnet, new_resnet)
content = content.replace(old_vgg, new_vgg)

with open('model/patnet.py', 'w') as f:
    f.write(content)

print("✓ Fixed weights enum")
!grep -n "weights" model/patnet.py

✓ Fixed weights enum
23:            self.backbone = vgg.vgg16(weights=vgg.VGG16_Weights.IMAGENET1K_V1)
37:            self.backbone = resnet.resnet50(weights=resnet.ResNet50_Weights.IMAGENET1K_V1)


In [41]:
import os
os.chdir('/kaggle/working/PATNet')

with open('train.py', 'r') as f:
    content = f.read()

# Add sys.stdout.flush() after write_process
content = content.replace(
    "average_meter.write_process(idx, len(dataloader), epoch, write_batch_idx=10)",
    "average_meter.write_process(idx, len(dataloader), epoch, write_batch_idx=10)\n        sys.stdout.flush()"
)

# Also add flush after write_result
content = content.replace(
    "average_meter.write_result('Training' if training else 'Validation', epoch)",
    "average_meter.write_result('Training' if training else 'Validation', epoch)\n    sys.stdout.flush()"
)

with open('train.py', 'w') as f:
    f.write(content)
print("✓ Added flush")

# Also check what write_process actually prints inside AverageMeter
with open('common/logger.py', 'r') as f:
    print(f.read())

✓ Added flush
r""" Logging during training/testing """
import datetime
import logging
import os

from tensorboardX import SummaryWriter
import torch


class AverageMeter:
    r""" Stores loss, evaluation results """
    def __init__(self, dataset):
        self.benchmark = dataset.benchmark
        if self.benchmark == 'pascal':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 20
        elif self.benchmark == 'fss':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 1000
        elif self.benchmark == 'deepglobe':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 6
        elif self.benchmark == 'isic':
            self.class_ids_interest = datas

In [42]:
import os
os.chdir('/kaggle/working/PATNet')

with open('common/logger.py', 'r') as f:
    content = f.read()

# Replace logging.info with print everywhere so Kaggle shows output
content = content.replace(
    "        Logger.info(msg)",
    "        print(msg, flush=True)\n        Logger.info(msg)"
)
content = content.replace(
    "    @classmethod\n    def info(cls, msg):\n        r\"\"\" Writes log message to log.txt \"\"\"\n        logging.info(msg)",
    "    @classmethod\n    def info(cls, msg):\n        r\"\"\" Writes log message to log.txt \"\"\"\n        print(msg, flush=True)\n        logging.info(msg)"
)

with open('common/logger.py', 'w') as f:
    f.write(content)
print("✓ Fixed logger to print to console")

✓ Fixed logger to print to console


In [45]:
import os
os.chdir('/kaggle/working/PATNet')

with open('common/logger.py', 'r') as f:
    content = f.read()

# Rewrite write_process cleanly
old = '''    def write_process(self, batch_idx, datalen, epoch, write_batch_idx=20):
        if batch_idx % write_batch_idx == 0:
            msg = '[Epoch: %02d] ' % epoch if epoch != -1 else ''
            msg += '[Batch: %04d/%04d] ' % (batch_idx+1, datalen)
            iou, fb_iou = self.compute_iou()
            if epoch != -1:
                loss_buf = torch.stack(self.loss_buf)
                msg += 'L: %6.5f  ' % loss_buf[-1]
                msg += 'Avg L: %6.5f  ' % loss_buf.mean()
            msg += 'mIoU: %5.2f  |  ' % iou
            msg += 'FB-IoU: %5.2f' % fb_iou
            print(msg, flus'''

# Find and replace everything from write_process to the next method
import re
new_write_process = '''    def write_process(self, batch_idx, datalen, epoch, write_batch_idx=20):
        if batch_idx % write_batch_idx == 0:
            msg = '[Epoch: %02d] ' % epoch if epoch != -1 else ''
            msg += '[Batch: %04d/%04d] ' % (batch_idx+1, datalen)
            iou, fb_iou = self.compute_iou()
            if epoch != -1:
                loss_buf = torch.stack(self.loss_buf)
                msg += 'L: %6.5f  ' % loss_buf[-1]
                msg += 'Avg L: %6.5f  ' % loss_buf.mean()
            msg += 'mIoU: %5.2f  |  ' % iou
            msg += 'FB-IoU: %5.2f' % fb_iou
            print(msg, flush=True)
            logging.info(msg)'''

# Replace the entire write_process method using regex
content = re.sub(
    r'    def write_process\(self.*?logging\.info\(msg\)',
    new_write_process,
    content,
    flags=re.DOTALL
)

with open('common/logger.py', 'w') as f:
    f.write(content)
print("✓ Fixed write_process")

# Verify
with open('common/logger.py', 'r') as f:
    text = f.read()
print(text[text.find('def write_process'):text.find('def write_process')+500])

✓ Fixed write_process
def write_process(self, batch_idx, datalen, epoch, write_batch_idx=20):
        if batch_idx % write_batch_idx == 0:
            msg = '[Epoch: %02d] ' % epoch if epoch != -1 else ''
            msg += '[Batch: %04d/%04d] ' % (batch_idx+1, datalen)
            iou, fb_iou = self.compute_iou()
            if epoch != -1:
                loss_buf = torch.stack(self.loss_buf)
                msg += 'L: %6.5f  ' % loss_buf[-1]
                msg += 'Avg L: %6.5f  ' % loss_buf.mean()
            msg


In [47]:
import os
os.chdir('/kaggle/working/PATNet')

logger_content = '''r""" Logging during training/testing """
import datetime
import logging
import os

from tensorboardX import SummaryWriter
import torch


class AverageMeter:
    r""" Stores loss, evaluation results """
    def __init__(self, dataset):
        self.benchmark = dataset.benchmark
        if self.benchmark == \'pascal\':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 20
        elif self.benchmark == \'fss\':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 1000
        elif self.benchmark == \'deepglobe\':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 6
        elif self.benchmark == \'isic\':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 3
        elif self.benchmark == \'lung\':
            self.class_ids_interest = dataset.class_ids
            self.class_ids_interest = torch.tensor(self.class_ids_interest).cuda()
            self.nclass = 1

        self.intersection_buf = torch.zeros([2, self.nclass]).float().cuda()
        self.union_buf = torch.zeros([2, self.nclass]).float().cuda()
        self.ones = torch.ones_like(self.union_buf)
        self.loss_buf = []

    def update(self, inter_b, union_b, class_id, loss):
        self.intersection_buf.index_add_(1, class_id, inter_b.float())
        self.union_buf.index_add_(1, class_id, union_b.float())
        if loss is None:
            loss = torch.tensor(0.0)
        self.loss_buf.append(loss)

    def compute_iou(self):
        iou = self.intersection_buf.float() / \
              torch.max(torch.stack([self.union_buf, self.ones]), dim=0)[0]
        iou = iou.index_select(1, self.class_ids_interest)
        miou = iou[1].mean() * 100

        fb_iou = (self.intersection_buf.index_select(1, self.class_ids_interest).sum(dim=1) /
                  self.union_buf.index_select(1, self.class_ids_interest).sum(dim=1)).mean() * 100

        return miou, fb_iou

    def write_result(self, split, epoch):
        iou, fb_iou = self.compute_iou()
        loss_buf = torch.stack(self.loss_buf)
        msg = \'\\n*** %s \' % split
        msg += \'[@Epoch %02d] \' % epoch
        msg += \'Avg L: %6.5f  \' % loss_buf.mean()
        msg += \'mIoU: %5.2f   \' % iou
        msg += \'FB-IoU: %5.2f   \' % fb_iou
        msg += \'***\\n\'
        print(msg, flush=True)
        logging.info(msg)

    def write_process(self, batch_idx, datalen, epoch, write_batch_idx=20):
        if batch_idx % write_batch_idx == 0:
            msg = \'[Epoch: %02d] \' % epoch if epoch != -1 else \'\'
            msg += \'[Batch: %04d/%04d] \' % (batch_idx+1, datalen)
            iou, fb_iou = self.compute_iou()
            if epoch != -1:
                loss_buf = torch.stack(self.loss_buf)
                msg += \'L: %6.5f  \' % loss_buf[-1]
                msg += \'Avg L: %6.5f  \' % loss_buf.mean()
            msg += \'mIoU: %5.2f  |  \' % iou
            msg += \'FB-IoU: %5.2f\' % fb_iou
            print(msg, flush=True)
            logging.info(msg)


class Logger:
    r""" Writes evaluation results of training/testing """
    @classmethod
    def initialize(cls, args, training):
        logtime = datetime.datetime.now().__format__(\'_%m%d_%H%M%S\')
        logpath = args.logpath if training else \'_TEST_\' + args.load.split(\'/\')[-2].split(\'.\')[0] + logtime
        if logpath == \'\': logpath = logtime

        cls.logpath = os.path.join(\'logs\', logpath + \'.log\')
        cls.benchmark = args.benchmark
        os.makedirs(cls.logpath)

        logging.basicConfig(filemode=\'w\',
                            filename=os.path.join(cls.logpath, \'log.txt\'),
                            level=logging.INFO,
                            format=\'%(message)s\',
                            datefmt=\'%m-%d %H:%M:%S\')

        console = logging.StreamHandler()
        console.setLevel(logging.INFO)
        formatter = logging.Formatter(\'%(message)s\')
        console.setFormatter(formatter)
        logging.getLogger(\'\').addHandler(console)

        cls.tbd_writer = SummaryWriter(os.path.join(cls.logpath, \'tbd/runs\'))

        logging.info(\'\\n:=========== Cross-Domain Few-shot Seg. with PATNet ===========\')
        for arg_key in args.__dict__:
            logging.info(\'| %20s: %-24s\' % (arg_key, str(args.__dict__[arg_key])))
        logging.info(\':================================================\\n\')

    @classmethod
    def info(cls, msg):
        print(msg, flush=True)
        logging.info(msg)

    @classmethod
    def save_model_miou(cls, model, epoch, val_miou):
        torch.save(model.state_dict(), os.path.join(cls.logpath, \'best_model.pt\'))
        msg = \'Model saved @%d w/ val. mIoU: %5.2f.\\n\' % (epoch, val_miou)
        print(msg, flush=True)
        logging.info(msg)

    @classmethod
    def log_params(cls, model):
        backbone_param = 0
        learner_param = 0
        for k in model.state_dict().keys():
            n_param = model.state_dict()[k].view(-1).size(0)
            if k.split(\'.\')[0] in \'backbone\':
                if k.split(\'.\')[1] in [\'classifier\', \'fc\']:
                    continue
                backbone_param += n_param
            else:
                learner_param += n_param
        Logger.info(\'Backbone # param.: %d\' % backbone_param)
        Logger.info(\'Learnable # param.: %d\' % learner_param)
        Logger.info(\'Total # param.: %d\' % (backbone_param + learner_param))
'''

with open('common/logger.py', 'w') as f:
    f.write(logger_content)
print("✓ logger.py fully rewritten")

✓ logger.py fully rewritten


In [51]:
import os
os.chdir('/kaggle/working/PATNet')

with open('train.py', 'r') as f:
    content = f.read()

content = content.replace('write_batch_idx=10)', 'write_batch_idx=100)')

with open('train.py', 'w') as f:
    f.write(content)
print("✓ Print frequency reduced")

✓ Print frequency reduced


In [58]:
import os, shutil
os.chdir('/kaggle/working/PATNet')
shutil.rmtree('/kaggle/working/PATNet/logs', ignore_errors=True)

with open('train.py', 'r') as f:
    content = f.read()

# Re-enable DataParallel correctly with the fixed model call
content = content.replace(
    "    model.to(device)",
    "    model = nn.DataParallel(model)\n    model.to(device)"
)
content = content.replace(
    "    model.train_mode() if training else model.eval()",
    "    model.module.train_mode() if training else model.module.eval()"
)
content = content.replace(
    "    loss = model.compute_objective(logit_mask, batch['query_mask'])",
    "    loss = model.module.compute_objective(logit_mask, batch['query_mask'])"
)

with open('train.py', 'w') as f:
    f.write(content)
print("✓ DataParallel re-enabled")
!grep -n "DataParallel\|module\." train.py

✓ DataParallel re-enabled
20:    model.module.train_mode() if training else model.module.eval()
28:        loss = model.module.compute_objective(logit_mask, batch['query_mask'])
60:    # Model initialization — no DataParallel, single GPU
65:    model = nn.DataParallel(model)


In [62]:
import torch, gc

# Free GPU memory
torch.cuda.empty_cache()
gc.collect()

# Confirm memory is free
for i in range(torch.cuda.device_count()):
    total = torch.cuda.get_device_properties(i).total_memory/1e9
    reserved = torch.cuda.memory_reserved(i)/1e9
    allocated = torch.cuda.memory_allocated(i)/1e9
    free = total - reserved
    print(f"GPU {i}: Total={total:.1f}GB  Allocated={allocated:.1f}GB  Free={free:.1f}GB")

GPU 0: Total=15.6GB  Allocated=0.0GB  Free=15.6GB
GPU 1: Total=15.6GB  Allocated=0.0GB  Free=15.6GB


In [65]:
import os
os.chdir('/kaggle/working/PATNet')
!grep -n "weights\|pretrained" model/patnet.py | head -5

23:            self.backbone = vgg.vgg16(weights=vgg.VGG16_Weights.IMAGENET1K_V1)
37:            self.backbone = resnet.resnet50(weights=resnet.ResNet50_Weights.IMAGENET1K_V1)


In [66]:
import os, shutil
os.chdir('/kaggle/working/PATNet')
shutil.rmtree('/kaggle/working/PATNet/logs', ignore_errors=True)

# Check dataset.py to see how episodes are controlled
with open('data/dataset.py', 'r') as f:
    print(f.read())

r""" Dataloader builder for few-shot semantic segmentation dataset  """
from torchvision import transforms
from torch.utils.data import DataLoader

from data.pascal import DatasetPASCAL
from data.fss import DatasetFSS
from data.deepglobe import DatasetDeepglobe
from data.isic import DatasetISIC
from data.lung import DatasetLung


class FSSDataset:

    @classmethod
    def initialize(cls, img_size, datapath):

        cls.datasets = {
            'pascal': DatasetPASCAL,
            'fss': DatasetFSS,
            'deepglobe': DatasetDeepglobe,
            'isic': DatasetISIC,
            'lung': DatasetLung,
        }

        cls.img_mean = [0.485, 0.456, 0.406]
        cls.img_std = [0.229, 0.224, 0.225]
        cls.datapath = datapath

        cls.transform = transforms.Compose([transforms.Resize(size=(img_size, img_size)),
                                            transforms.ToTensor(),
                                            transforms.Normalize(cls.img_mean, cls.img_std)])


In [67]:
with open('data/pascal.py', 'r') as f:
    print(f.read())

r""" PASCAL-5i few-shot semantic segmentation dataset """
import os

from torch.utils.data import Dataset
import torch.nn.functional as F
import torch
import PIL.Image as Image
import numpy as np


class DatasetPASCAL(Dataset):
    def __init__(self, datapath, fold, transform, split, shot):
        self.split = 'val' if split in ['val', 'test'] else 'trn'
        self.fold = fold
        self.nfolds = 4
        self.nclass = 20
        self.benchmark = 'pascal'
        self.shot = shot

        self.img_path = os.path.join('/kaggle/working/VOC2012/JPEGImages/')
        self.ann_path = os.path.join('/kaggle/working/VOC2012/SegmentationClassAug/')
        self.transform = transform

        self.class_ids = self.build_class_ids()
        self.img_metadata = self.build_img_metadata()
        self.img_metadata_classwise = self.build_img_metadata_classwise()

    def __len__(self):
        return len(self.img_metadata) if self.split == 'trn' else 1000

    def __getitem__(self, idx):
      

In [69]:
import os, shutil
os.chdir('/kaggle/working/PATNet')
shutil.rmtree('/kaggle/working/PATNet/logs', ignore_errors=True)

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py \
    --backbone resnet50 \
    --fold 0 \
    --benchmark pascal \
    --lr 1e-3 \
    --bsz 12 \
    --niter 50 \
    --nworker 0 \
    --logpath patnet_fold0

Backbone # param.: 23561205
Learnable # param.: 2580968
Total # param.: 26142173
# available GPUs: 2
Total (trn) images are : 11394
Total (val) images are : 346
[Epoch: 00] [Batch: 0001/0167] L: 0.68500  Avg L: 0.68500  mIoU:  0.00  |  FB-IoU: 35.12
[Epoch: 00] [Batch: 0101/0167] L: 0.48220  Avg L: 0.50533  mIoU:  5.40  |  FB-IoU: 49.08

*** Training [@Epoch 00] Avg L: 0.45697  mIoU:  7.97   FB-IoU: 56.61   ***

[Epoch: 00] [Batch: 0001/0017] L: 0.37318  Avg L: 0.37318  mIoU: 36.92  |  FB-IoU: 59.60

*** Validation [@Epoch 00] Avg L: 0.34992  mIoU: 47.85   FB-IoU: 67.14   ***

Model saved @0 w/ val. mIoU: 47.85.

  >> New best model saved! val_mIoU: 47.85
[Epoch: 01] [Batch: 0001/0167] L: 0.33202  Avg L: 0.33202  mIoU: 12.77  |  FB-IoU: 73.15
[Epoch: 01] [Batch: 0101/0167] L: 0.43575  Avg L: 0.35705  mIoU: 12.05  |  FB-IoU: 69.85

*** Training [@Epoch 01] Avg L: 0.34991  mIoU: 12.24   FB-IoU: 70.41   ***

[Epoch: 01] [Batch: 0001/0017] L: 0.27945  Avg L: 0.27945  mIoU: 44.71  |  FB-IoU